<a href="https://colab.research.google.com/github/Swingandamiss/ML_projects/blob/main/podium_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')
df = pd.merge(results, races[['raceId', 'year', 'round', 'circuitId', 'name']], on='raceId', how='left')
print(df.head())

   resultId  raceId  driverId  constructorId number  grid position  \
0         1      18         1              1     22     1        1   
1         2      18         2              2      3     5        2   
2         3      18         3              3      7     7        3   
3         4      18         4              4      5    11        4   
4         5      18         5              1     23     3        5   

  positionText  positionOrder  points  ...  milliseconds fastestLap rank  \
0            1              1    10.0  ...       5690616         39    2   
1            2              2     8.0  ...       5696094         41    3   
2            3              3     6.0  ...       5698779         41    5   
3            4              4     5.0  ...       5707797         58    7   
4            5              5     4.0  ...       5708630         43    1   

  fastestLapTime fastestLapSpeed statusId  year  round  circuitId  \
0       1:27.452         218.300        1  2008      

In [ ]:
drivers = pd.read_csv('drivers.csv')
constructors = pd.read_csv('constructors.csv')
status = pd.read_csv('status.csv')
df = pd.merge(df, drivers[['driverId', 'driverRef', 'dob']], on='driverId', how='left')
constructors = constructors.rename(columns={'name': 'constructor_name'})
df = pd.merge(df, constructors[['constructorId', 'constructor_name']], on='constructorId', how='left')
df = pd.merge(df, status[['statusId', 'status']], on='statusId', how='left')
# Clean up the columns for readability
df = df.rename(columns={'name': 'race_name', 'driverRef': 'driver_name'})
print(f"Dataset size: {df.shape[0]} rows, {df.shape[1]} columns")
print(df[['year', 'race_name', 'driver_name', 'constructor_name', 'grid', 'positionOrder', 'status']].head(10))

Dataset size: 26759 rows, 26 columns
   year              race_name driver_name constructor_name  grid  \
0  2008  Australian Grand Prix    hamilton          McLaren     1   
1  2008  Australian Grand Prix    heidfeld       BMW Sauber     5   
2  2008  Australian Grand Prix     rosberg         Williams     7   
3  2008  Australian Grand Prix      alonso          Renault    11   
4  2008  Australian Grand Prix  kovalainen          McLaren     3   
5  2008  Australian Grand Prix    nakajima         Williams    13   
6  2008  Australian Grand Prix    bourdais       Toro Rosso    17   
7  2008  Australian Grand Prix   raikkonen          Ferrari    15   
8  2008  Australian Grand Prix      kubica       BMW Sauber     2   
9  2008  Australian Grand Prix       glock           Toyota    18   

   positionOrder     status  
0              1   Finished  
1              2   Finished  
2              3   Finished  
3              4   Finished  
4              5   Finished  
5              6     +1

In [ ]:
# Filter out the ancient history
df = df[df['year'] >= 2014].copy()

# Create our Target Variable: The Podium!
df['podium'] = df['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)

# Sort the data chronologically so it makes sense
df = df.sort_values(by=['year', 'round', 'positionOrder'])

print("Total modern races to train on:", df['raceId'].nunique())
print(df[['year', 'driver_name', 'constructor_name', 'positionOrder', 'podium']].head(10))

Total modern races to train on: 228
       year      driver_name constructor_name  positionOrder  podium
22127  2014          rosberg         Mercedes              1       1
22128  2014  kevin_magnussen          McLaren              2       1
22129  2014           button          McLaren              3       1
22130  2014           alonso          Ferrari              4       0
22131  2014           bottas         Williams              5       0
22132  2014       hulkenberg      Force India              6       0
22133  2014        raikkonen          Ferrari              7       0
22134  2014           vergne       Toro Rosso              8       0
22135  2014            kvyat       Toro Rosso              9       0
22136  2014            perez      Force India             10       0


In [ ]:
# Sort the data by driver, then chronologically by year and round
df = df.sort_values(by=['driverId', 'year', 'round'])

# Calculate the rolling 3-race average of points for each driver
df['driver_form'] = df.groupby('driverId')['points'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# If a driver is a rookie (or it's race 1), they won't have a past 3 races. Fill those blanks with 0.
df['driver_form'] = df['driver_form'].fillna(0)

# Let's check our work for a specific driver, like Max Verstappen (driverId 830)
max_data = df[df['driver_name'] == 'hamilton'][['year', 'race_name', 'points', 'driver_form']]
print(max_data.tail(10))

       year                 race_name  points  driver_form
26566  2024          Dutch Grand Prix     4.0    21.666667
26583  2024        Italian Grand Prix    10.0    14.666667
26607  2024     Azerbaijan Grand Prix     2.0    13.000000
26624  2024      Singapore Grand Prix     8.0     5.333333
26658  2024  United States Grand Prix     0.0     6.666667
26662  2024    Mexico City Grand Prix    12.0     3.333333
26688  2024      São Paulo Grand Prix     1.0     6.666667
26700  2024      Las Vegas Grand Prix    18.0     4.333333
26730  2024          Qatar Grand Prix     0.0    10.333333
26742  2024      Abu Dhabi Grand Prix    12.0     6.333333


In [ ]:
print(df['driver_name'].unique())

['hamilton' 'rosberg' 'alonso' 'raikkonen' 'kubica' 'massa' 'sutil'
 'button' 'vettel' 'grosjean' 'kobayashi' 'hulkenberg' 'maldonado' 'resta'
 'perez' 'ricciardo' 'vergne' 'chilton' 'gutierrez' 'bottas'
 'jules_bianchi' 'kevin_magnussen' 'kvyat' 'lotterer' 'ericsson' 'stevens'
 'max_verstappen' 'nasr' 'sainz' 'merhi' 'rossi' 'jolyon_palmer'
 'wehrlein' 'haryanto' 'vandoorne' 'ocon' 'stroll' 'giovinazzi' 'gasly'
 'brendon_hartley' 'leclerc' 'sirotkin' 'norris' 'russell' 'albon'
 'latifi' 'pietro_fittipaldi' 'aitken' 'tsunoda' 'mazepin'
 'mick_schumacher' 'zhou' 'de_vries' 'piastri' 'sargeant' 'lawson'
 'bearman' 'colapinto' 'doohan']


In [ ]:
df['constructor_form'] = df.groupby('constructorId')['points'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
df['constructor_form'] = df['constructor_form'].fillna(0)
features = ['grid', 'driver_form', 'constructor_form']
target = 'podium'

# ----- THE TRAIN / TEST SPLIT ---
# We train the model on history (2014 to 2022)
train_df = df[df['year'] < 2022]
# We test the model on the 2023 season
test_df = df[df['year'] >= 2023]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print(f"Training on {len(X_train)} races. Testing on {len(X_test)} races.\n")

# ----- IMPORT THE ALGORITHM ---
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix

# ----- INITIALIZE AND TRAIN ---
model = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=4)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("--- MODEL PERFORMANCE ---")
print(f"Overall Accuracy: {accuracy_score(y_test, predictions):.2f}")
print(f"Precision Score:  {precision_score(y_test, predictions):.2f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Training on 3267 races. Testing on 919 races.

--- MODEL PERFORMANCE ---
Overall Accuracy: 0.90
Precision Score:  0.69

Confusion Matrix:
[[742  39]
 [ 53  85]]
